# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Naveed-Qasim608/Flyrank_ML_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Baseline Rule

The baseline ranks pages using a simple hand-written score.

Rule:

Pages receive a higher priority score when they:
- Have high search impressions (visibility)
- Have not been updated recently
- Show signs of declining performance

Reason codes:

STALE_VISIBLE:
Page is old but still receives search visibility.

HIGH_EXPOSURE:
Page has many impressions and represents a larger opportunity.

DECLINING_SIGNAL:
Page shows observed decline in search performance.

This baseline is transparent and provides a fair comparison for the ML model.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv(
    "data/raw/content_refresh_anonymized.csv"
)

# Create baseline score

stale = (
    df["days_since_last_update"] >= 180
).astype(int)

visible = (
    df["impressions_90d"] >= 500
).astype(int)


df["baseline_score"] = (
    stale *
    visible *
    df["impressions_90d"]
)


# Reason codes

def reason(row):
    if row["baseline_score"] > 0:
        return "STALE_VISIBLE"
    return "LOW_PRIORITY"


df["reason_code"] = df.apply(reason, axis=1)


# Rank queue

queue = df.sort_values(
    "baseline_score",
    ascending=False
)


os.makedirs(
    "work/outputs",
    exist_ok=True
)


queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)


print("Saved baseline queue")
print(queue.head(10))

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 Review

The top-ranked pages represent pages where the baseline found a combination of:

- High visibility
- Long time since update

Action:

Review the page content, check search intent, and decide whether a refresh is needed.

Confidence note:

Confidence is based only on observed signals from the dataset.

Possible reasons it could be wrong:

- The page may already be performing well.
- The content may not need updating.
- External factors may explain performance changes.

In [ ]:
top20 = queue.head(20)

top20[
    [
        "impressions_90d",
        "days_since_last_update",
        "avg_position",
        "ctr",
        "reason_code"
    ]
]

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks and Leakage Check

Some recommendations may be incorrect because:

- Old content does not always mean poor quality.
- High impressions do not guarantee improvement opportunity.

Leakage check:

The baseline does not use:
- trend_direction
- trend_pct
- future performance data
- product decision flags

Only information available before review is used.

In [ ]:
leak_columns = [
    "trend_direction",
    "trend_pct"
]


for col in leak_columns:
    print(
        col,
        "USED" if col in queue.columns else "NOT USED"
    )

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.